In [1]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")
sys.path.append("/Users/bubble/Desktop/Project/T_sensor/T_sensor")

import gdsfactory as gf
# TODO
# 1. Change one side of the cell_temp to be comparation with the width
# 2. Different with the width and same width with gap
from Tools.geo_trans import round_corner

cell_temp = gf.Component()

2026-06-02 13:10:46.652 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/1015314453.oas'
2026-06-02 13:26:08.932 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/2561479770.oas'
2026-06-02 13:27:25.589 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/430415169.oas'
2026-06-02 13:32:05.130 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/2956498146.oas'
2026-06-02 13:32:53.205 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/1366731409.oas'


In [ ]:
# bottom left
L1 = 900 # length of beams
w1 = 5 # width of beam
gap = 50 # gap between beams
L2 = 500 # length of hinges
w2 = [0.5, 0.5, 0.5] # width of hinges
n_beam = [1, 3, 9]  # number of beams

p = 9
grid = [None] * p
for i in range(p):
    grid[i] = gf.Component()
block = gf.Component()
count = 0
temp_T = gf.Component()
offset = 6
for i in range(len(n_beam)):
    for j in range(len(w2)):
            row = count // 3
            col = count % 3
            total_height = L2*2 + n_beam[i]*w1 + (n_beam[i]-1)*gap

            # optical lithography layer (Solid beam / thinn down)
            opt_beam = gf.components.rectangle(size=(L1, w1), layer=(8, 0))
            for k in range(n_beam[i]):
                (grid[count] << opt_beam).move((-L1/2, L2+ k*(w1 + gap)))
            h_connector = w1*n_beam[i]+gap*(n_beam[i]-1)
            opt_connector = gf.components.rectangle(size=(w1, h_connector), layer=(8, 0))
            for k in range(3):
                if k == 1:
                    (grid[count] << opt_connector).move((L1/2*(k-1)-w1/2, L2))
                else:
                    (grid[count] << opt_connector).move((L1/2*(k-1), L2))

            # Thin down layer
            thin_down_beam = gf.components.rectangle(size=(L1+offset, n_beam[i]*w1 + (n_beam[i]-1)*gap+offset), layer=(10, 0))
            (grid[count] << thin_down_beam).move((-(L1+offset)/2, L2-offset/2))
            thin_down_hinge = gf.components.rectangle(size=(w2[j]+offset, L2), layer=(10, 0))
            (grid[count] << thin_down_hinge).move((-(w2[j] + offset)/2, 0))
            (grid[count] << thin_down_hinge).move((-(w2[j] + offset)/2, total_height - L2))


            # frontside frame
            opt_frame = gf.components.rectangle(size=(1000, total_height), layer=(9, 0))
            (block << opt_frame).move((col * 2000, row * 2000))
            (block << grid[count]).move((500+col * 2000, row * 2000))

            # Ebeam lithography layer
            T_hinge = gf.components.rectangle(size=(w2[j], L2), layer=(5, 0))
            (grid[count] << T_hinge).move((-w2[j]/2, 0))
            (grid[count] << T_hinge).move((-w2[j]/2, total_height - L2))
            T_corner = round_corner(4*w2[j], w2[j], 2*w2[j], rotation=90, layer=(5, 0))
            (grid[count] << T_corner)
            (grid[count] << T_corner).dmirror_y(L2/2)
            (grid[count] << T_corner).dmirror_y(total_height/2)
            (grid[count] << T_corner).dmirror_y(L2/2).dmirror_y(total_height/2)

            T_beam = gf.components.rectangle(size=(L1, w1), layer=(5, 0))
            for k in range(n_beam[i]):
                (grid[count] << T_beam).move((-L1/2, L2+ k*(w1 + gap)))
            h_connector = w1*n_beam[i]+gap*(n_beam[i]-1)
            T_connector = gf.components.rectangle(size=(w1, h_connector), layer=(5, 0))
            for k in range(3):
                if k == 1:
                    (grid[count] << T_connector).move((L1/2*(k-1)-w1/2, L2))
                else:
                    (grid[count] << T_connector).move((L1/2*(k-1), L2))
            offset = 10
            frame_hinge = gf.components.rectangle(size=(w2[j]+2*offset, L2-offset), layer=(6, 0))
            frame_beam = gf.components.rectangle(size=(L1+2*offset, h_connector+2*offset), layer=(6, 0))
            (grid[count] << frame_hinge).move((-(w2[j] + 2*offset)/2, 0))
            (grid[count] << frame_hinge).move((-(w2[j] + 2*offset)/2, total_height - (L2-offset)))
            (grid[count] << frame_beam).move((-(L1+2* offset)/2, L2-offset))

            # ebeam frame
            frame_hinge = gf.components.rectangle(size=(w2[j]+2*offset, L2-offset), layer=(6, 0))
            (grid[count] << frame_hinge).move((-(w2[j] + 2*offset)/2, 0))
            (grid[count] << frame_hinge).move((-(w2[j] + 2*offset)/2, total_height - (L2-offset)))

            # backside etching layer
            size_x = 1743.44
            size_y = total_height+743.44
            diff_x = (size_x - 1000) / 2  
            diff_y = (size_y - total_height) / 2
            backside = gf.components.rectangle(size=(size_x, size_y), layer=(3, 0)) 
            (block << backside).move((col * 2000 - diff_x, row * 2000 - diff_y))

            # length mark
            mark_text = gf.components.text(f"L1={L1} w1={w1} l2={L2} w2={w2[j]} n={n_beam[i]}", size=40, layer=(1, 0))
            (block << mark_text).move((col * 2000, row * 2000-200))
            count += 1

# block.show()




In [18]:
# structure for each die
# structure for 4 sides
fblock = gf.Component()
diff = 445/2
for i in range(4):
    if i == 0:
        block_ref = fblock << block
        block_ref.move((0, -diff))
    elif i == 1:
        block_ref = fblock << block
        block_ref.move((10000, -diff))
    elif i == 2:
        block_ref = fblock << block
        block_ref.move((10000, 10000-diff))
    else:
        block_ref = fblock << block
        block_ref.move((0, 10000-diff))

In [19]:
# order
order = gf.Component()
text_array = []
for i in range(4):
    text_array.append(gf.Component())

for i in range(4):
    T = gf.components.text(f"TMB{i+1}", size=20, layer=(1, 0))
    for j in range(4):
        order_ref = text_array[i] << T
        if j == 0:
            order_ref.move((-80, 0))
        elif j == 1:
            order_ref.move((-80, 5000))
        elif j == 2:
            order_ref.move((5020, 0))
        else:
            order_ref.move((5020, 5000))
    text_array_ref = order << text_array[i]
    if i == 0:
        pass
    elif i == 1:
        text_array_ref.move((10000, 0))
    elif i == 2:
        text_array_ref.move((0, 10000))
    else:
        text_array_ref.move((10000, 10000))


In [ ]:
# boolean operation
# thin down layer
thin_down_layer = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(9, 0), layer2=(10, 0), layer=(7, 0))
cell_temp << thin_down_layer

# optical lithography layer
outside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(9, 0), layer2=(8, 0), layer=(1, 0))
cell_temp << outside

# Ebeam
T_structure = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(6, 0), layer2=(5, 0), layer=(5, 0))
cell_temp << T_structure

# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(15, 0), layer=(1, 0))
cell_temp << marker
cell_temp << order

# add backside etching
backside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(3, 0), layer2=(100, 0), layer=(3, 0))
cell_temp << backside

# frame
frame1 = gf.components.rectangle(size=(15000, 15000), layer=(20, 0))
frame2 = gf.components.rectangle(size=(20000, 20000), layer=(21, 0))
frame2_ref = cell_temp << frame2
frame2_ref.move((-2500, -2500))
cell_temp << frame1

cell_T_multy_beam_L900_500 = gf.Component()
cell_ref = cell_T_multy_beam_L900_500 << cell_temp
cell_ref.move((2500, -7500))
cell_T_multy_beam_L900_500.show()
# cell_T_multy_beam_L900_500.write_gds("mesh.gds")
# cell_T_multy_beam_L900_500.plot()